<style>
.jp-Notebook, .notebook-container, .markdown-body {font-family: Arial, Helvetica, sans-serif;}
.jp-MarkdownOutput, .text_cell_render {font-size: 18px; line-height: 1.65;}
h1 {font-size: 2.25rem !important; margin-top: 0.35em !important;}
h2 {font-size: 1.65rem !important; margin-top: 1.35em !important;}
h3 {font-size: 1.25rem !important; margin-top: 1.1em !important;}
table {font-size: 0.95em;}
blockquote {border-left: 4px solid #aaa; padding-left: 1rem;}
</style>


# 04 · Pruebas de hipótesis

<p><a href="https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/notebooks/04_Pruebas_de_Hipotesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a></p>

> **Objetivo:** convertir una afirmación en una prueba interpretable sin confundir significancia con importancia práctica.


<img src="https://raw.githubusercontent.com/mauriciorslrv/DS_basics/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/imgs/statisticaltesttree.png" alt="Árbol de pruebas estadísticas" width="780">


## 1. La pregunta va antes que la prueba

Pregunta:

> **¿un proceso nuevo reduce el tiempo promedio de entrega?**

- **H₀:** el proceso nuevo no reduce el promedio.
- **H₁:** el proceso nuevo tiene un promedio menor.

### 💡 IDEA
Una prueba no "demuestra" H₀ o H₁. Evalúa qué tan compatibles son los datos con H₀ bajo ciertos supuestos.


### ¿Por qué este ejercicio?
Usamos dos grupos sintéticos porque conocemos el mecanismo que los generó. Eso permite concentrarnos en **la lógica de inferencia**: observar → cuantificar diferencia → estimar incertidumbre → interpretar.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng=np.random.default_rng(123)

actual=rng.normal(38,7,60)
nuevo=rng.normal(34.5,7,60)


## 2. Mira los datos antes del p-value

Una prueba estadística no sustituye una inspección básica. Primero observamos posición, dispersión y posibles valores extremos.


In [ ]:
plt.boxplot([actual,nuevo],labels=["Actual","Nuevo"])
plt.ylabel("Tiempo (min)")
plt.show()

print("Media actual:", actual.mean())
print("Media nuevo:", nuevo.mean())
print("Diferencia observada:", nuevo.mean()-actual.mean())


## 3. Welch t-test

Usamos **Welch** porque compara medias de dos muestras independientes sin exigir varianzas iguales.

Como nuestra hipótesis alternativa es direccional (`nuevo < actual`), calculamos un p-value **unilateral**. También mostramos el bilateral para comparar.


In [ ]:
res_unilateral = stats.ttest_ind(nuevo, actual, equal_var=False, alternative="less")
res_bilateral = stats.ttest_ind(nuevo, actual, equal_var=False, alternative="two-sided")

print(f"t = {res_unilateral.statistic:.3f}")
print(f"p-value unilateral = {res_unilateral.pvalue:.4f}")
print(f"p-value bilateral = {res_bilateral.pvalue:.4f}")


### El p-value NO es

- la probabilidad de que H₀ sea verdadera;
- la probabilidad de que "todo sea azar";
- el tamaño del efecto;
- una medida automática de importancia práctica.

Interpreta siempre el p-value junto con **diseño, tamaño del efecto e incertidumbre**.


## 4. Tamaño de efecto

### ¿Por qué?
Una diferencia puede ser estadísticamente detectable pero muy pequeña en términos prácticos. Cohen's *d* expresa la diferencia en unidades de desviación estándar.


In [ ]:
n1,n2=len(nuevo),len(actual)
s1,s2=nuevo.std(ddof=1),actual.std(ddof=1)

sp=np.sqrt(((n1-1)*s1**2+(n2-1)*s2**2)/(n1+n2-2))
d=(nuevo.mean()-actual.mean())/sp

print(f"Cohen's d = {d:.3f}")


## 5. Intervalo de confianza para la diferencia

### ¿Por qué?
El intervalo de confianza muestra un **rango de valores compatibles con los datos y el método**. Nos ayuda a pensar en magnitud e incertidumbre, no sólo en superar un umbral.


In [ ]:
diff=nuevo.mean()-actual.mean()

se=np.sqrt(nuevo.var(ddof=1)/n1+actual.var(ddof=1)/n2)

df_num=(nuevo.var(ddof=1)/n1+actual.var(ddof=1)/n2)**2
df_den=((nuevo.var(ddof=1)/n1)**2)/(n1-1)+((actual.var(ddof=1)/n2)**2)/(n2-1)
df_welch=df_num/df_den

crit=stats.t.ppf(.975,df_welch)
ci=(diff-crit*se,diff+crit*se)

print(f"Diferencia media: {diff:.2f} min")
print(f"IC 95%: ({ci[0]:.2f}, {ci[1]:.2f})")


<img src="https://raw.githubusercontent.com/mauriciorslrv/DS_basics/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/imgs/Distribuciones_Hipotesis.png" alt="Distribuciones e hipótesis" width="700">


## 6. Supuestos y diseño

Antes de interpretar, pregunta por:

- independencia;
- sesgo de selección;
- tamaño de muestra;
- outliers;
- medición consistente;
- pruebas múltiples;
- si la hipótesis fue definida antes de mirar los resultados.

### ⚠️ ERROR ÚTIL
`p < 0.05` no convierte un análisis débil en uno bueno. El diseño y la magnitud del efecto importan.


## 7. Mapa, no catálogo

<img src="https://raw.githubusercontent.com/mauriciorslrv/DS_basics/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/imgs/ANOVA.png" alt="ANOVA" width="640">

No memorices un árbol completo de pruebas de inmediato. Primero identifica:

**tipo de variable → número de grupos → independencia/dependencia → pregunta → supuestos**

Después eliges la herramienta.


## 8. Mini reto

Formula H₀ y H₁ para:

> "el nuevo proceso reduce el tiempo promedio por debajo de 20 minutos".

Indica si la alternativa es unilateral o bilateral y qué cambio considerarías **relevante en la práctica**, aunque el p-value fuese pequeño.


## 📚 Material adicional
- [SciPy · Hypothesis tests](https://docs.scipy.org/doc/scipy/tutorial/stats/hypothesis_tests.html) — ejemplos de pruebas y comparaciones.
- [ASA · Statement on Statistical Significance and P-Values](https://www.amstat.org/asa/files/pdfs/p-valuestatement.pdf) — interpretación responsable del p-value.
- [NIST · Comparing means](https://www.itl.nist.gov/div898/handbook/prc/section3/prc31.htm) — pruebas de medias y supuestos.

### 🧾 Términos clave
**H₀ · H₁ · p-value · nivel de significancia · tamaño de efecto · intervalo de confianza · error estándar · potencia**


## Qué sigue
Monte Carlo nos permitirá explorar **miles de escenarios posibles** cuando varias entradas son inciertas.
